# Building blocks 01 — one rack branch, then one rack row

This notebook walks through the new `blocks` package step by step:

1. Heat load and maximum temperature rise → mass flow and volume flow
2. Volume flow and velocity cap → smallest catalogue pipe that fits
3. One **rack branch**: every component, its pressure drop, the total and the head
4. One **rack row**: headers + 8 branches, balanced so every rack gets its design flow
5. What the row asks of whatever feeds it (the connection interface)

Flows are **prescribed** from heat and temperature rise. Nothing here solves a flow network. A pressure drop that has not been supplied (the rack's internal loop) is shown as **unknown**, not zero.

In [1]:
import sys
from pathlib import Path
from dataclasses import replace

ENGINE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ENGINE))

from blocks import Fluid, RackBranchSpec, RowSpec, flow_from_heat, size_pipe, build_rack_branch, build_row
from blocks.row import DESCHUTES_IT_DP_AVAILABLE, PSI

try:
    import pandas as pd
    pd.set_option('display.precision', 3)
    def show(rows, cols=None):
        df = pd.DataFrame(rows)
        return df[cols] if cols else df
except ImportError:                     # pandas is optional
    def show(rows, cols=None):
        for r in rows:
            print({k: r[k] for k in (cols or r)})

## 0. Coolant properties

There is no built-in PG25 table on purpose: properties depend on temperature, glycol fraction and product. Here we use the same values the rest of the engine uses (`model.Config` defaults), which are **project inputs** to be confirmed against the manufacturer's property curve.

In [2]:
from model import Config
fluid = Fluid.from_config(Config())
fluid.as_dict()

{'name': 'TCS coolant (engine configuration)',
 'rho_kg_m3': 1025.0,
 'cp_J_kgK': 3900.0,
 'mu_Pa_s': 0.002,
 'source': 'Engine configuration (model.Config); project input at the intended mean temperature. Formulation and manufacturer property curve require confirmation.',
 'status': 'project_input'}

## 1. Flow from heat and maximum temperature rise

ṁ = Q / (cp · ΔT),  V̇ = ṁ / ρ

In [3]:
RACK_HEAT_W = 350_000.     # liquid heat into the coolant at one rack
DT_MAX_K    = 10.          # maximum coolant rise across the rack
SUPPLY_T_C  = 30.

d = flow_from_heat(RACK_HEAT_W, DT_MAX_K, fluid)
print(f'mass flow   {d.mass_kg_s:.3f} kg/s')
print(f'volume flow {d.flow_m3_s*1000:.3f} L/s = {d.flow_L_min:.1f} L/min = {d.flow_L_min/3.785411784:.1f} gpm')

mass flow   8.974 kg/s
volume flow 8.755 L/s = 525.3 L/min = 138.8 gpm


## 2. Pipe size from the velocity cap

The cap and the catalogue belong to the **pipe category** in `standards.py`. The rack-branch category is Type L copper with a 1.5 m/s cap (marked *assumption*); the row header is Sch 10S stainless at 2.7 m/s (CloudScale §3.4, *recommended*).

In [4]:
rows = []
for cat in ('tcs_rack_branch', 'tcs_row_header'):
    p = size_pipe(d.flow_m3_s, cat)
    rows.append({'category': cat, 'catalogue': p['catalogue'], 'cap_m_s': p['velocity_cap_m_s'],
                 'cap_status': p['velocity_cap_status'], 'min_ID_mm': p['required_id_m']*1000,
                 'selected_NPS_in': p['nominal_size_in'], 'ID_mm': p['id_m']*1000, 'velocity_m_s': p['velocity_m_s']})
show(rows)

,category,catalogue,cap_m_s,cap_status,min_ID_mm,selected_NPS_in,ID_mm,velocity_m_s
0,tcs_rack_branch,copper_type_l,1.5,assumption,86.208,3.5,86.995,1.473
1,tcs_row_header,stainless_sch10,2.7,recommended,64.256,2.5,66.929,2.489


## 3. One rack branch

Path: isolation valve → strainer → supply pipe + elbows → quick disconnect → **rack** → quick disconnect → return pipe + elbows → balancing valve → isolation valve.

Lengths, elbow counts and the 10 kPa balancing-valve minimum are declared assumptions until the branch is routed and a valve is chosen.

In [5]:
branch_spec = RackBranchSpec(heat_W=RACK_HEAT_W, delta_T_max_K=DT_MAX_K, supply_T_C=SUPPLY_T_C)
branch = build_rack_branch(branch_spec, fluid)
show(branch.element_rows(), ['tag', 'kind', 'flow_L_min', 'velocity_m_s', 'k', 'dp_kPa', 'status'])

,tag,kind,flow_L_min,velocity_m_s,k,dp_kPa,status
0,R01-IV-S,isolation_valve,525.328,1.473,0.2,0.222,assumption
1,R01-STR,strainer,525.328,1.473,2.0,2.224,assumption
2,R01-P-S,pipe,525.328,1.473,NaN,0.757,calculated
3,R01-EL-S1,elbow,525.328,1.473,0.6,0.667,assumption
4,R01-EL-S2,elbow,525.328,1.473,0.6,0.667,assumption
5,R01-QD-S,quick_disconnect,525.328,1.473,2.0,2.224,assumption
6,R01-RACK,rack_load,525.328,NaN,NaN,NaN,unknown
7,R01-QD-R,quick_disconnect,525.328,1.473,2.0,2.224,assumption
8,R01-P-R,pipe,525.328,1.473,NaN,0.757,calculated
9,R01-EL-R1,elbow,525.328,1.473,0.6,0.667,assumption


In [6]:
branch.summary()

{'name': 'Rack branch R01',
 'kind': 'rack_branch',
 'heat_kW': 350.0,
 'mass_kg_s': 8.974358974358974,
 'flow_L_min': 525.328330206379,
 'supply_T_C': 30.0,
 'return_T_C': 40.0,
 'delta_T_K': 10.0,
 'dp_kPa': 21.299868376212288,
 'head_m': 2.1190069382945884,
 'dp_complete': False,
 'unknowns': ['R01-RACK'],
 'failed_checks': []}

The total is **incomplete** because the rack's own pressure drop (manifold + cold plates) has not been supplied. Supplying it — here a placeholder value, *not* vendor data — completes the branch:

In [7]:
PLACEHOLDER_RACK_DP_PA = 100_000.   # placeholder for illustration only; replace with vendor data
branch_known = build_rack_branch(replace(branch_spec, rack_dp_Pa=PLACEHOLDER_RACK_DP_PA,
                                         rack_rated_flow_m3_s=d.flow_m3_s,
                                         rack_dp_source='PLACEHOLDER for illustration, not vendor data'), fluid)
{k: branch_known.summary()[k] for k in ('dp_kPa', 'head_m', 'dp_complete')}

{'dp_kPa': 121.29986837621229,
 'head_m': 12.06745779661584,
 'dp_complete': True}

## 4. One rack row: 8 racks, four header options

Every rack gets its design flow. The path with the largest pressure drop is the critical one; every other branch's balancing valve takes up the difference so all paths need the same pressure at the row connection.

* **direct return** — supply enters and return leaves at the same end; the farthest rack is critical.
* **reverse return** — return header leaves at the far end and a return leg comes back, so paths are nearly equal.
* **constant** header — one size for the full row flow; **stepped** — each segment sized for its own flow.

In [8]:
N = 8
variants = []
for arrangement in ('direct_return', 'reverse_return'):
    for header in ('constant', 'stepped'):
        r = build_row(RowSpec(branch_spec, N, arrangement=arrangement, header=header,
                              available_dp=DESCHUTES_IT_DP_AVAILABLE), fluid)
        bv = [v['dp_kPa'] for v in r.valves if v['kind'] == 'balancing_valve']
        variants.append({'arrangement': arrangement, 'header': header,
                         'row_flow_L_min': r.flow_L_min, 'feed_NPS_in': r.pipes[0]['nominal_size_in'],
                         'row_dp_kPa (known parts)': r.dp_kPa, 'head_m': r.head_m,
                         'BV min/max kPa': f'{min(bv):.1f} / {max(bv):.1f}', 'complete': r.dp_complete})
show(variants)

,arrangement,header,row_flow_L_min,feed_NPS_in,row_dp_kPa (known parts),head_m,BV min/max kPa,complete
0,direct_return,constant,4202.627,8,34.717,3.454,10.0 / 17.1,False
1,direct_return,stepped,4202.627,8,56.909,5.662,10.0 / 39.3,False
2,reverse_return,constant,4202.627,8,34.271,3.409,10.0 / 12.9,False
3,reverse_return,stepped,4202.627,8,43.394,4.317,10.0 / 12.8,False


### Balancing valve schedule — direct return, constant header

In [9]:
row = build_row(RowSpec(branch_spec, N, arrangement='direct_return', header='constant',
                       available_dp=DESCHUTES_IT_DP_AVAILABLE), fluid)
show(row.valves, ['tag', 'kind', 'size_nominal_in', 'flow_L_min', 'dp_kPa', 'Kv_m3_h', 'Cv_US'])

,tag,kind,size_nominal_in,flow_L_min,dp_kPa,Kv_m3_h,Cv_US
0,Row A-IV-S,isolation_valve,8.0,4202.627,0.407,NaN,NaN
1,Row A-IV-R,isolation_valve,8.0,4202.627,0.407,NaN,NaN
2,Row A-R01-IV-S,isolation_valve,3.5,525.328,0.222,NaN,NaN
3,Row A-R01-BV,balancing_valve,3.5,525.328,17.062,77.255,89.307
4,Row A-R01-IV-R,isolation_valve,3.5,525.328,0.222,NaN,NaN
5,Row A-R02-IV-S,isolation_valve,3.5,525.328,0.222,NaN,NaN
6,Row A-R02-BV,balancing_valve,3.5,525.328,14.734,83.135,96.104
7,Row A-R02-IV-R,isolation_valve,3.5,525.328,0.222,NaN,NaN
8,Row A-R03-IV-S,isolation_valve,3.5,525.328,0.222,NaN,NaN
9,Row A-R03-BV,balancing_valve,3.5,525.328,12.982,88.567,102.383


### Header pipe schedule

In [10]:
show(row.pipes, ['run', 'length_m', 'flow_L_min', 'nominal_size_in', 'velocity_m_s', 'velocity_cap_m_s', 'velocity_cap_status'])

,run,length_m,flow_L_min,nominal_size_in,velocity_m_s,velocity_cap_m_s,velocity_cap_status
0,Row A header S0,5.000,4202.627,8.0,1.993,2.7,recommended
1,Row A header S1,0.716,3677.298,8.0,1.744,2.7,recommended
2,Row A header S2,0.716,3151.970,8.0,1.494,2.7,recommended
3,Row A header S3,0.716,2626.642,8.0,1.245,2.7,recommended
4,Row A header S4,0.716,2101.313,8.0,0.996,2.7,recommended
5,Row A header S5,0.716,1575.985,8.0,0.747,2.7,recommended
6,Row A header S6,0.716,1050.657,8.0,0.498,2.7,recommended
7,Row A header S7,0.716,525.328,8.0,0.249,2.7,recommended
8,Row A header R0,5.000,4202.627,8.0,1.993,2.7,recommended
9,Row A header R1,0.716,3677.298,8.0,1.744,2.7,recommended


## 5. What the row asks of its feeder

This dictionary is the interface for the next block (the CDU / distribution). The feeder does not need the row's internals.

In [11]:
row.connection()

{'block': 'Row A',
 'kind': 'rack_row',
 'mass_kg_s': 71.7948717948718,
 'flow_m3_s': 0.07004377736085053,
 'supply_T_C': 30.0,
 'return_T_C': 40.0,
 'required_dp_Pa': 34716.84481639383,
 'required_head_m': 3.453788246118589,
 'dp_complete': False,
 'unknowns': ['Row A-R01-RACK',
  'Row A-R02-RACK',
  'Row A-R03-RACK',
  'Row A-R04-RACK',
  'Row A-R05-RACK',
  'Row A-R06-RACK',
  'Row A-R07-RACK',
  'Row A-R08-RACK']}

In [12]:
show(row.checks, ['name', 'value', 'limit', 'unit', 'passed'])

,name,value,limit,unit,passed
0,velocity Row A header S0,1.993,2.700e+00,m/s,True
1,velocity Row A header S1,1.744,2.700e+00,m/s,True
2,velocity Row A header S2,1.494,2.700e+00,m/s,True
3,velocity Row A header S3,1.245,2.700e+00,m/s,True
4,velocity Row A header S4,0.996,2.700e+00,m/s,True
5,velocity Row A header S5,0.747,2.700e+00,m/s,True
6,velocity Row A header S6,0.498,2.700e+00,m/s,True
7,velocity Row A header S7,0.249,2.700e+00,m/s,True
8,velocity Row A header R0,1.993,2.700e+00,m/s,True
9,velocity Row A header R1,1.744,2.700e+00,m/s,True


The available-pressure check against the OCP Deschutes CDU (80 psi, §3.1) shows **None** — it cannot pass while the rack pressure drop is unknown. It only applies if the row is fed by a Deschutes-class CDU.

In [13]:
row.assumptions

['Branch straight lengths 3 m supply / 3 m return with 2/2 elbows (project assumption until routed).',
 'Balancing valve minimum allocation 10 kPa at the critical branch (project assumption; vendor minimum measuring/signal differential to be confirmed).',
 'Rack internal pressure drop not supplied: branch total is incomplete.',
 'Row feed length 5 m from the row connection to the first tee (project assumption).',
 'Rack pitch 0.7162 m.',
 'Tee, reducer and isolation-valve K values are preliminary_sizing.DEFAULT_K screening assumptions.',
 'Every rack receives its design flow (prescribed); balancing valves are allocated to make that so.']

## 6. Sensitivity to the allowed temperature rise

A larger rise means less flow, smaller pipes and lower pressure drop — at the cost of a hotter return.

In [14]:
rows = []
for dT in (6., 8., 10., 12., 15.):
    s = replace(branch_spec, delta_T_max_K=dT)
    r = build_row(RowSpec(s, N, arrangement='reverse_return'), fluid)
    rows.append({'dT_K': dT, 'rack_L_min': r.flow_L_min / N, 'branch_NPS_in': r.children[0].pipes[0]['nominal_size_in'],
                 'header_NPS_in': r.pipes[0]['nominal_size_in'], 'return_T_C': r.return_T_C,
                 'row_dp_kPa (known)': r.dp_kPa})
show(rows)

,dT_K,rack_L_min,branch_NPS_in,header_NPS_in,return_T_C,row_dp_kPa (known)
0,6.0,875.547,5.0,10,36.0,29.441
1,8.0,656.660,4.0,8,38.0,37.812
2,10.0,525.328,3.5,8,40.0,34.271
3,12.0,437.774,3.5,8,42.0,26.957
4,15.0,350.219,3.0,6,45.0,35.737
